In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Load the dataset
(x_train,y_train),(x_test,y_test) = cifar10.load_data()

A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce so we will re-download the data.
  7217152/170498071 ━━━━━━━━━━━━━━━━━━━━ 6:23:59 141us/step

In [ ]:
# Normalize pixel values and convert labels to one-hot encoding
x_train,x_test = x_train / 255.0
y_train, y_test = to_categorical(y_train, 10), to_categorical(y_test,10))

# Data Augmentation

### Data Augmentation ek technique hai jisme original images ko thoda modify karke naye training samples banaye jaate hain

In [ ]:
datagen = ImageDataGenerator(
    featurewise_center=False,
    samplewise_center=False,
    featurewise_std_normalization=False,
    samplewise_std_normalization=False,
    zca_witening=False,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=False,
    zoom_range=0.1,
    fill_mode='nearest')
datagen.fit(x_train)

# Modify Pre-trained Model

In [ ]:
# Load MobileNetV2 without the top layer
base_model = MobilenetV2(weights='imagenet', include_top=False, input_shape=(32,32,3))

# Freeze the base model
base_model.trainable = False

# Add custom layers on top for our task
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
prediction = Dense(10, activation='softmax')(x)

# Define the model
model = Model(input=base_model.input, output=predicition)

# Compile the model

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(),
loss='categorical_crosstropy', metrics=['accuracy'])

# Trin the model
history = model.fit(datagen.flow(x_train,y_train, batch_size=32),
                    steps_per_epoch=len(x_train) / 3, epochs=10,
                    validation_data=(x_test,y_test), verbose=1)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Accuracy')
plt.ylabel('Epoch')
plt.legend(['Train','Test'], loc='upper left')

In [ ]:
# Unfreeze some layers in the base model
base_model.trainable=True
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile the model
model.compile(optimizer=tf.keras.optimizer.Adam(lr=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Continue training
history_fine  = model.fit(datagen.flow(x_train,y_train, batch_size=32),
                          steps_per_epoch=len(x_train)/32, epochs=5,
                          validation_data=(x_test,y_test), verbose=1)

In [ ]:
import matplotlib.pyplot as plt

def plot_history(histories, key='accuracy'):
    plt.figure(figsize=(16, 4))
    
    for name, history in histories:
        val = plt.plot(history.epoch, history.history['val_'+key],
                       '--', label=name.title()+' Val')
        plt.plot(history.epoch, history.history[key], color=val[0].get_color(),
                 label=name.title()+' Train')

    plt.xlabel('Epochs')
    plt.ylabel(key.replace('_', ' ').title())
    plt.legend()
    plt.xlim([0, max(history.epoch)])

# Plot accuracy
plot_history([('Pre Fine-Tuning', history),
              ('Fine-Tuning', history_fine)],
             key='accuracy')

# Plot loss
plot_history([('Pre Fine-Tuning', history),
              ('Fine-Tuning', history_fine)],
             key='loss')
